<!--nav--> [🗺 Learning path](README.md) · **35/44** · ◀ [Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb) · [Long-Context Serving](./LongContext_KV_Compression_Serving.ipynb) ▶

# The What-If Console: One Model for Every Serving Decision

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Serving_WhatIf_Console.ipynb)

You now have ten notebooks of mechanisms and two of hardware. This one **consolidates all of it into
a single predictive model** you can interrogate:

> *What if we moved to MI300X? What if we quantized to FP8? What if the SLO tightened to 30ms?
> What if traffic tripled? What if prefix hit rate went from 20% to 60%?*

Answering those in a meeting, with numbers, is the skill this whole track was building toward.

| Part | What you'll get |
|---|---|
| **1** | The consolidated model — every equation, sourced to the notebook that derived it |
| **2** | **Calibration**: checking the model against the measurements from the earlier notebooks |
| **3** | **The console** — an interactive what-if across GPU, precision, batch, context, TP, speculation |
| **4** | **Tornado sensitivity**: which knob actually moves your cost |
| **5** | **The Pareto frontier**: every config, cost vs latency, with the dominated ones greyed out |
| **6** | Six worked what-if scenarios, answered end to end |
| **7** | Honest error bars — where this model lies to you |

**Runs on:** any CPU. This notebook is pure modeling; the measurements it's calibrated against came
from the GPU notebooks earlier in the track.

In [ ]:
import math, json, uuid, itertools
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · The consolidated model

Every line below is traceable to a notebook that derived or measured it. That's what makes this a
model rather than a spreadsheet of guesses.

| Quantity | Formula | Source |
|---|---|---|
| KV bytes/token | `2 × layers × kv_heads × head_dim × kv_bytes` | [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb) |
| KV pool | `VRAM × util − weights − overhead` | [Reading the Logs](./Serving_Logs_Observability.ipynb) (startup log) |
| Max concurrency | `kv_pool_tokens / avg_context` | [Reading the Logs](./Serving_Logs_Observability.ipynb) |
| Decode (memory-bound) | `BW_eff / (params × weight_bytes) × batch` | [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb), [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) |
| Decode (compute-bound) | `FLOPS_eff / (2 × params)` | [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) (roofline) |
| Flip point | `B* = ridge × weight_bytes / 2` | [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) |
| Prefill | `FLOPS_eff / (2 × params)` tokens/s | [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb), [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) |
| Effective prefill | `× (1 − prefix_hit_rate)` | [vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb) |
| Speculation gain | `(1 − α^(k+1))/(1−α) / (k·c + 1)` | [Speculative Decoding](./Speculative_Decoding_Advanced_Serving.ipynb) |
| TP speedup | `1 / (1/N + comm·(N−1)/N)` | [Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb) |
| Goodput | requests/s meeting TTFT *and* TPOT SLOs | [Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb) |
| $/1M tokens | `(GPU $/hr ÷ 3600) ÷ tok/s × 10⁶` | [Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb) |

In [ ]:
# ---------------------------------------------------------------- hardware & model catalogs
GPUS = {
  #                    vendor    VRAM  BW TB/s fp16TF fp8TF  $/hr  fp8?  int4-tuned?
  "T4":         dict(vendor="NVIDIA", vram=16,  bw=0.32, tf16=65,   tf8=None, usd=0.35, int4=0.55),
  "L4":         dict(vendor="NVIDIA", vram=24,  bw=0.30, tf16=121,  tf8=242,  usd=0.70, int4=0.90),
  "A10G":       dict(vendor="NVIDIA", vram=24,  bw=0.60, tf16=125,  tf8=None, usd=1.00, int4=0.90),
  "A100 80GB":  dict(vendor="NVIDIA", vram=80,  bw=2.04, tf16=312,  tf8=None, usd=2.20, int4=0.95),
  "H100 SXM":   dict(vendor="NVIDIA", vram=80,  bw=3.35, tf16=990,  tf8=1979, usd=3.50, int4=0.95),
  "H200 SXM":   dict(vendor="NVIDIA", vram=141, bw=4.80, tf16=990,  tf8=1979, usd=4.50, int4=0.95),
  "B200":       dict(vendor="NVIDIA", vram=192, bw=8.00, tf16=2250, tf8=4500, usd=7.00, int4=0.95),
  "MI210":      dict(vendor="AMD",    vram=64,  bw=1.60, tf16=181,  tf8=None, usd=1.20, int4=0.45),
  "MI250X":     dict(vendor="AMD",    vram=128, bw=3.20, tf16=383,  tf8=None, usd=2.20, int4=0.45),
  "MI300X":     dict(vendor="AMD",    vram=192, bw=5.30, tf16=1307, tf8=2615, usd=3.50, int4=0.70),
  "MI325X":     dict(vendor="AMD",    vram=256, bw=6.00, tf16=1307, tf8=2615, usd=4.20, int4=0.70),
  "MI355X":     dict(vendor="AMD",    vram=288, bw=8.00, tf16=2300, tf8=4600, usd=7.00, int4=0.75),
}
# 'int4' = kernel-quality factor for weight-only int4 (portable-kernels's maturity gap), 1.0 = ideal.

MODELS = {
  "Qwen2.5-0.5B": dict(params=0.5, layers=24, kv_heads=2,  head_dim=64),
  "Llama-3.1-8B": dict(params=8.0, layers=32, kv_heads=8,  head_dim=128),
  "Qwen2.5-32B":  dict(params=32.0,layers=64, kv_heads=8,  head_dim=128),
  "Llama-3.1-70B":dict(params=70.0,layers=80, kv_heads=8,  head_dim=128),
}
PRECISION = {   # weight bytes, whether it needs fp8 hw, whether it's the int4 path
  "fp16": dict(bytes=2.0,  needs_fp8=False, int4=False),
  "fp8":  dict(bytes=1.0,  needs_fp8=True,  int4=False),
  "int4": dict(bytes=0.5,  needs_fp8=False, int4=True),
}

BW_EFF, FLOP_EFF, MEM_UTIL = 0.75, 0.60, 0.90     # achievable fractions (roofline Part 7 measures these)
MIN_CONCURRENCY = 32       # a KV pool too small to hold this many requests isn't a viable deployment
ENGINE_OVERHEAD_MS = 0.8   # fixed per-decode-step cost of an efficient engine (CUDA/HIP graphs, vLLM)

def predict(gpu_name, model_name, precision="fp16", batch=32, ctx=2048,
            prefix_hit=0.0, spec_alpha=0.0, spec_k=4, spec_cost=0.05,
            tp=None, kv_fp8=False, overhead_ms=ENGINE_OVERHEAD_MS):
    # Predict serving performance for one configuration.
    # Returns a dict with an "error" key if the configuration cannot run at all.
    g, m, p = GPUS[gpu_name], MODELS[model_name], PRECISION[precision]
    if p["needs_fp8"] and not g["tf8"]:
        return {"error": f"{gpu_name} has no FP8 hardware (portable-kernels) - would be emulated"}

    weights_gb = m["params"] * p["bytes"]
    kv_bytes = 1 if kv_fp8 else 2
    kv_per_tok = 2 * m["layers"] * m["kv_heads"] * m["head_dim"] * kv_bytes
    # A deployment needs weights AND a KV pool big enough to be worth running (logs):
    kv_needed_gb = kv_per_tok * ctx * MIN_CONCURRENCY / 1e9

    # --- topology: shard until weights + a USABLE KV pool fit (this is why long ctx forces wide TP) ---
    if tp is None:
        tp = 1
        while tp < 16 and (weights_gb + kv_needed_gb) / tp > g["vram"] * MEM_UTIL:
            tp *= 2
    if (weights_gb + kv_needed_gb) / tp > g["vram"] * MEM_UTIL:
        return {"error": f"needs >{16} GPUs to hold weights + a {MIN_CONCURRENCY}-request KV pool at {ctx} ctx"}
    tp_eff = 1.0 / (1.0 / tp + 0.06 * (tp - 1) / tp) if tp > 1 else 1.0   # the distributed-serving notebook

    # --- KV pool (serving-fundamentals, logs) ---
    pool_gb = g["vram"] * tp * MEM_UTIL - weights_gb - 1.5 * tp
    max_conc = max(0, int(pool_gb * 1e9 // (kv_per_tok * ctx)))
    eff_batch = min(batch, max_conc) if max_conc else 0
    if eff_batch == 0:
        return {"error": f"KV pool cannot hold even one {ctx}-token request"}

    # --- decode: the roofline (the roofline notebook) PLUS the engine's fixed per-step cost ---
    bw = g["bw"] * 1e12 * BW_EFF * tp * tp_eff
    peak = (g["tf8"] if precision == "fp8" else g["tf16"]) * 1e12 * FLOP_EFF * tp * tp_eff
    kernel = g["int4"] if p["int4"] else 1.0            # the portable-kernels notebook maturity factor
    mem_step_s  = (m["params"] * 1e9 * p["bytes"]) / bw / kernel     # time to stream the weights once
    comp_step_s = (2 * m["params"] * 1e9 * eff_batch) / peak         # time to do the batch's math
    # One decode step serves the whole batch; overhead is per step, not per request.
    step_s = max(mem_step_s, comp_step_s) + overhead_ms / 1000
    decode_tps = eff_batch / step_s
    bound = ("overhead" if overhead_ms / 1000 > max(mem_step_s, comp_step_s)
             else "memory" if mem_step_s > comp_step_s else "compute")

    # --- speculation (the speculation notebook) ---
    spec_mult = 1.0
    if spec_alpha > 0:
        a, k = spec_alpha, spec_k
        spec_mult = ((1 - a ** (k + 1)) / (1 - a)) / (k * spec_cost + 1)
        spec_mult = max(1.0, spec_mult * (1.0 if bound == "memory" else 0.4))   # less spare compute
    decode_tps *= spec_mult

    per_req_tps = decode_tps / eff_batch
    tpot_ms = 1000 / max(per_req_tps, 1e-9)

    # --- prefill (serving-fundamentals, vLLM) ---
    prefill_tps = peak / (2 * m["params"] * 1e9)
    ttft_s = ctx * (1 - prefix_hit) / max(prefill_tps, 1e-9)

    gpus_used = tp
    cost_hr = g["usd"] * gpus_used
    cpm = (cost_hr / 3600) / max(decode_tps, 1e-9) * 1e6

    return {"gpu": gpu_name, "vendor": g["vendor"], "model": model_name, "precision": precision,
            "tp": tp, "gpus": gpus_used, "batch": eff_batch, "max_concurrency": max_conc,
            "decode_tps": decode_tps, "per_req_tps": per_req_tps, "tpot_ms": tpot_ms,
            "ttft_ms": ttft_s * 1000, "prefill_tps": prefill_tps, "bound": bound,
            "spec_mult": spec_mult, "kv_pool_gb": pool_gb, "usd_hr": cost_hr, "cpm": cpm}

r = predict("H100 SXM", "Llama-3.1-8B", "fp16", batch=32)
print("example prediction — Llama-3.1-8B, fp16, batch 32, H100:")
for k, v in r.items():
    print(f"  {k:<18}{v if not isinstance(v, float) else round(v, 2)}")

## Part 2 · Calibration — and being honest about what we actually know

A model nobody checked is a rumor. But there are **two very different kinds of check**, and mixing
them up is how planning models end up trusted more than they deserve:

1. **Identity checks** — things the model must get right *by construction*, because they're
   arithmetic derived in earlier notebooks (KV bytes/token, the flip point, TP topology from VRAM).
   These must pass exactly. If one fails, the model has a bug.
2. **Ballpark checks** — comparisons against publicly reported figures for real deployments. These
   get wide tolerance because published numbers vary with engine version, prompt shape, and how
   generous the reporter felt.

> **What this notebook does *not* claim:** the GPU-only cells in the measured notebooks need hardware this
> environment doesn't have, so **no measured numbers from those runs are baked in here**. When you
> run them on a real T4 (or anything else), Part 2b shows you how to fold your own measurements in —
> that's the calibration that actually makes this model yours.

In [ ]:
# --- 2a. IDENTITY CHECKS: arithmetic the model must reproduce exactly (serving-fundamentals, distributed-serving, roofline) --------
identity = []

# KV bytes/token for Llama-3.1-8B GQA, fp16 — the serving-fundamentals notebook computed 128 KB/token
m = MODELS["Llama-3.1-8B"]
kv_kb = 2 * m["layers"] * m["kv_heads"] * m["head_dim"] * 2 / 1024
identity.append(("KV/token, 8B GQA fp16 (the serving-fundamentals notebook)", kv_kb, 128.0, "KB"))

# Flip point in fp16 equals the ridge point (the roofline notebook): B* = ridge x bytes / 2, bytes=2
g = GPUS["H100 SXM"]
ridge = g["tf16"] * 1e12 / (g["bw"] * 1e12)
identity.append(("H100 ridge = fp16 flip point (roofline)", ridge * 2 / 2, ridge, "FLOP/byte"))

# TP=4 at 6% comm: speedup 1/(1/4 + 0.06*3/4) = 3.39x on 4 GPUs -> 84.7% efficiency (distributed-serving)
tp4 = 1.0 / (1 / 4 + 0.06 * 3 / 4) / 4
identity.append(("TP=4 efficiency at 6% comm (the distributed-serving notebook)", tp4, 0.8475, "fraction"))

print("2a · IDENTITY CHECKS (must be exact)")
print(f"{'check':<42}{'model':>12}{'expected':>12}{'':>4}")
print("-" * 74)
ident_ok = 0
for label, got, want, unit in identity:
    ok = abs(got - want) / max(abs(want), 1e-9) < 0.01
    ident_ok += ok
    print(f"{label:<42}{got:>12.2f}{want:>12.2f} {unit:<10}{'✓' if ok else '✗ BUG'}")

# --- 2b. TOPOLOGY CHECKS: does the fit logic pick the TP width practitioners actually use? ----
print("\n2b · TOPOLOGY CHECKS (weights + a usable KV pool must fit)")
topo = [
    ("70B fp16 on 80GB cards -> TP=4",  dict(gpu_name="H100 SXM", model_name="Llama-3.1-70B",
                                             batch=8, ctx=2048), 4),
    ("70B fp16 on MI300X 192GB -> TP=1", dict(gpu_name="MI300X", model_name="Llama-3.1-70B",
                                              batch=8, ctx=2048), 1),
    ("8B fp16 on A100 80GB -> TP=1",     dict(gpu_name="A100 80GB", model_name="Llama-3.1-8B",
                                              batch=8, ctx=2048), 1),
]
print(f"{'case':<42}{'model TP':>10}{'expected':>10}")
print("-" * 64)
topo_ok = 0
for label, cfg, want in topo:
    got = predict(**cfg)["tp"]
    topo_ok += (got == want)
    print(f"{label:<42}{got:>10}{want:>10}   {'✓' if got == want else '✗'}")

# --- 2c. BALLPARK CHECKS: public reference points, wide tolerance, clearly labelled -----------
print("\n2c · BALLPARK CHECKS vs publicly reported ranges (±40%, illustrative only)")
ballpark = [
    ("8B fp16, A100 80GB, batch 32", dict(gpu_name="A100 80GB", model_name="Llama-3.1-8B",
                                          batch=32, ctx=2048), 2600),
    ("8B fp16, H100 SXM, batch 32",  dict(gpu_name="H100 SXM", model_name="Llama-3.1-8B",
                                          batch=32, ctx=2048), 4300),
]
print(f"{'case':<42}{'predicted':>11}{'reference':>11}{'error':>9}")
print("-" * 74)
ball_ok = 0
for label, cfg, ref in ballpark:
    got = predict(**cfg)["decode_tps"]
    err = abs(got - ref) / ref
    ball_ok += err <= 0.40
    print(f"{label:<42}{got:>11,.0f}{ref:>11,.0f}{err:>8.0%}   {'✓' if err <= 0.40 else '✗'}")

print(f"\nidentity {ident_ok}/{len(identity)} · topology {topo_ok}/{len(topo)} · "
      f"ballpark {ball_ok}/{len(ballpark)}")
print("\nIdentity and topology checks are the ones that matter for correctness.")
print("Ballpark checks only say 'not obviously wrong' - never quote them as a prediction.")

In [ ]:
# --- 2d. Fold in YOUR measurements ------------------------------------------------------------
# Run the roofline notebook Part 7 on your hardware, then set these to what you actually measured.
def recalibrate(measured_bw_tbs, measured_gemm_tf, gpu_name, overhead_ms=None):
    '''Replace the model's efficiency assumptions with numbers measured on YOUR box.'''
    g = GPUS[gpu_name]
    bw_eff = measured_bw_tbs / g["bw"]
    flop_eff = measured_gemm_tf / g["tf16"]
    print(f"{gpu_name}: measured {measured_bw_tbs:.2f} TB/s of {g['bw']:.2f} peak -> BW_EFF {bw_eff:.2f}")
    print(f"{'':<{len(gpu_name)}}  measured {measured_gemm_tf:.0f} TF of {g['tf16']} peak -> FLOP_EFF {flop_eff:.2f}")
    if bw_eff > 1 or flop_eff > 1:
        print("  ⚠ measured above peak - check your benchmark, not your GPU")
    return bw_eff, flop_eff

print("Example: a T4 that measured 0.24 TB/s and 48 TFLOP/s\n")
bw_eff, flop_eff = recalibrate(0.24, 48, "T4")
print(f"\nThen set the globals and re-run the notebook:")
print(f"  BW_EFF, FLOP_EFF = {bw_eff:.2f}, {flop_eff:.2f}")
print("\nAlso worth calibrating: ENGINE_OVERHEAD_MS. Measure it as the decode step time at")
print("batch 1 minus (weight bytes / measured bandwidth). Frameworks differ by ~5x here -")
print("HF eager is overhead-dominated at small batch; vLLM with graph capture is not (vLLM).")

## Part 3 · The console

Everything above, made interactive. Change any input; every output updates, including which
constraint is binding and what the model recommends you do next.

In [ ]:
# Precompute the full configuration grid so the browser can explore it instantly.
grid = []
for gpu in GPUS:
    for model in MODELS:
        for prec in PRECISION:
            for batch in (1, 4, 16, 32, 64, 128, 256):
                for ctx in (1024, 4096, 16384):
                    r = predict(gpu, model, prec, batch=batch, ctx=ctx)
                    if "error" in r:
                        continue
                    grid.append({k: (round(v, 3) if isinstance(v, float) else v)
                                 for k, v in r.items()
                                 if k in ("gpu","vendor","model","precision","tp","gpus","batch",
                                          "decode_tps","tpot_ms","ttft_ms","bound","cpm",
                                          "max_concurrency","usd_hr")}
                                | {"req_batch": batch, "ctx": ctx})
print(f"precomputed {len(grid):,} feasible configurations")
print(f"  GPUs: {len(GPUS)} · models: {len(MODELS)} · precisions: {len(PRECISION)}")
infeasible = len(GPUS)*len(MODELS)*len(PRECISION)*7*3 - len(grid)
print(f"  {infeasible:,} combinations were infeasible (no FP8 hardware, or won't fit)")

In [ ]:
JS = r'''
const sel = root.append("div").style("font","13px system-ui").style("margin-bottom","8px");
function dropdown(label, values) {
  const w = sel.append("label").style("margin-right","16px");
  w.append("span").text(label + " ");
  const s = w.append("select").style("font-size","13px");
  s.selectAll("o").data(values).join("option").attr("value",d=>d).text(d=>d);
  s.on("change", draw);
  return s;
}
const uniq = k => [...new Set(data.map(d => d[k]))];
const gpuSel  = dropdown("GPU:", uniq("gpu"));
const modSel  = dropdown("model:", uniq("model"));
const precSel = dropdown("precision:", uniq("precision"));
const ctxSel  = dropdown("context:", uniq("ctx").sort((a,b)=>a-b));
modSel.property("value", "Llama-3.1-8B");
gpuSel.property("value", "H100 SXM");

const out = root.append("div").style("font","13px ui-monospace,monospace")
    .style("background","#f6f8fa").style("padding","10px 12px").style("border-radius","8px")
    .style("white-space","pre-wrap").style("margin-bottom","10px");

const M = {top: 18, right: 60, bottom: 42, left: 60};
const iw = W - M.left - M.right, ih = 210;
const svg = root.append("svg").attr("width",W).attr("height",ih+M.top+M.bottom)
    .append("g").attr("transform",`translate(${M.left},${M.top})`);
const x = d3.scaleLog().range([0, iw]);
const y = d3.scaleLog().range([ih, 0]);
const y2 = d3.scaleLog().range([ih, 0]);
const xAxis = svg.append("g").attr("transform",`translate(0,${ih})`);
const yAxis = svg.append("g");
const yAxis2 = svg.append("g").attr("transform",`translate(${iw},0)`);
svg.append("text").attr("x",iw/2).attr("y",ih+34).attr("text-anchor","middle")
   .style("font-size","11.5px").text("batch size (requests in flight)");
svg.append("text").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-44)
   .attr("text-anchor","middle").style("font-size","11.5px").style("fill","#1976d2")
   .text("decode tokens/s");
svg.append("text").attr("transform","rotate(90)").attr("x",ih/2).attr("y",-iw-44)
   .attr("text-anchor","middle").style("font-size","11.5px").style("fill","#d32f2f")
   .text("TPOT (ms)");
const tpsPath = svg.append("path").attr("fill","none").attr("stroke","#1976d2").attr("stroke-width",2.5);
const tpotPath = svg.append("path").attr("fill","none").attr("stroke","#d32f2f").attr("stroke-width",2.5);
const flipLine = svg.append("line").attr("y1",0).attr("y2",ih).attr("stroke","#333")
    .attr("stroke-dasharray","5 4");
const flipTxt = svg.append("text").style("font-size","10.5px").style("font-weight",600);
const dots = svg.append("g");

function draw() {
  const gpu = gpuSel.property("value"), mod = modSel.property("value");
  const prec = precSel.property("value"), ctx = +ctxSel.property("value");
  const rows = data.filter(d => d.gpu===gpu && d.model===mod && d.precision===prec && d.ctx===ctx)
                   .sort((a,b)=>a.req_batch-b.req_batch);
  if (!rows.length) {
    out.text(`❌ infeasible: ${mod} in ${prec} does not fit on ${gpu} at ${ctx} context ` +
             `(no FP8 hardware, or not enough VRAM even with tensor parallelism).`);
    tpsPath.attr("d",null); tpotPath.attr("d",null); dots.selectAll("*").remove();
    flipTxt.text(""); flipLine.attr("x1",-10).attr("x2",-10);
    return;
  }
  x.domain(d3.extent(rows, d=>d.req_batch));
  y.domain([d3.min(rows,d=>d.decode_tps)*0.8, d3.max(rows,d=>d.decode_tps)*1.3]);
  y2.domain([d3.min(rows,d=>d.tpot_ms)*0.7, d3.max(rows,d=>d.tpot_ms)*1.4]);
  xAxis.call(d3.axisBottom(x).tickValues(rows.map(d=>d.req_batch)).tickFormat(d3.format("d")));
  yAxis.call(d3.axisLeft(y).ticks(4,"~s"));
  yAxis2.call(d3.axisRight(y2).ticks(4,"~s"));
  tpsPath.datum(rows).attr("d", d3.line().x(d=>x(d.req_batch)).y(d=>y(d.decode_tps)));
  tpotPath.datum(rows).attr("d", d3.line().x(d=>x(d.req_batch)).y(d=>y2(d.tpot_ms)));
  dots.selectAll("c").data(rows).join("circle")
      .attr("cx",d=>x(d.req_batch)).attr("cy",d=>y(d.decode_tps)).attr("r",4)
      .attr("fill",d=>d.bound==="memory" ? "#1976d2" : "#ff9800")
      .select(function(){return this;});
  dots.selectAll("c").selectAll("title").remove();
  dots.selectAll("c").append("title")
      .text(d=>`batch ${d.req_batch}: ${d3.format(",.0f")(d.decode_tps)} tok/s · ` +
               `TPOT ${d.tpot_ms.toFixed(1)}ms · ${d.bound}-bound · $${d.cpm.toFixed(3)}/1M`);
  const flip = rows.find(d=>d.bound==="compute");
  if (flip) {
    flipLine.attr("x1",x(flip.req_batch)).attr("x2",x(flip.req_batch));
    flipTxt.attr("x",Math.min(x(flip.req_batch)+5, iw-215)).attr("y",14)
           .text(`flips to compute-bound ≈ batch ${flip.req_batch}`);
  } else {
    flipLine.attr("x1",-10).attr("x2",-10);
    flipTxt.attr("x",6).attr("y",14).text("memory-bound at every batch shown");
  }
  const best = rows.reduce((a,b)=>a.cpm<b.cpm?a:b);
  const r32 = rows.find(d=>d.req_batch===32) || rows[rows.length-1];
  out.text(
`${mod} · ${prec} · ${gpu} (${r32.vendor}) · ${ctx} ctx
topology      : TP=${r32.tp}  (${r32.gpus} GPU${r32.gpus>1?"s":""} per replica)  ·  KV holds ${r32.max_concurrency} concurrent requests
at batch ${String(r32.req_batch).padEnd(4)}: ${d3.format(",.0f")(r32.decode_tps)} tok/s total · TPOT ${r32.tpot_ms.toFixed(1)}ms · TTFT ${r32.ttft_ms.toFixed(0)}ms · ${r32.bound}-bound
cheapest here : batch ${best.req_batch} → $${best.cpm.toFixed(3)} per 1M output tokens
next move     : ${r32.bound === "memory"
   ? "memory-bound → quantize weights, speculative decoding, or a higher-bandwidth GPU (quantization, speculation, roofline)"
   : "compute-bound → FP8/FP4, better kernels, or more GPUs; further weight quantization won't help (the roofline notebook)"}`);
}
draw();
'''
show_d3(JS, grid, height=300)

**Things worth trying in the console:**

- **`Llama-3.1-70B` + `fp16` + `H100 SXM`** → TP=4. Now switch the GPU to **`MI300X`** → **TP=1**.
  Same model, same precision; the topology collapsed because of VRAM alone ([The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) Part 6).
- **`fp8` on `A100 80GB`** → infeasible, and the console says why: no FP8 hardware ([Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb)).
- Any model on **`T4`** → memory-bound at *every* batch size shown. On **`B200`**, the same model
  flips to compute-bound much earlier. Same code, different bottleneck — that's the porting problem
  in one picture.
- **`int4` on `MI210` vs `A100`** → note the cost difference beyond raw specs: the model applies
  [Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb)'s int4 kernel-maturity factor.

## Part 4 · Which knob actually matters? (tornado)

Sensitivity analysis: from one baseline, move each variable independently and measure the effect on
**$ per million tokens**. The longest bar is where your attention belongs.

In [ ]:
BASE = dict(gpu_name="H100 SXM", model_name="Llama-3.1-8B", precision="fp16",
            batch=32, ctx=4096, prefix_hit=0.0, spec_alpha=0.0)
base = predict(**BASE)
print(f"baseline: {BASE['model_name']} on {BASE['gpu_name']}, fp16, batch 32, 4k ctx")
print(f"          {base['decode_tps']:,.0f} tok/s · TPOT {base['tpot_ms']:.1f}ms · "
      f"${base['cpm']:.3f}/1M tokens\n")

VARIATIONS = [
    ("batch 32 → 128",            dict(batch=128)),
    ("batch 32 → 8",              dict(batch=8)),
    ("fp16 → fp8",                dict(precision="fp8")),
    ("fp16 → int4",               dict(precision="int4")),
    ("H100 → MI300X",             dict(gpu_name="MI300X")),
    ("H100 → A100 80GB",          dict(gpu_name="A100 80GB")),
    ("H100 → B200",               dict(gpu_name="B200")),
    ("H100 → MI355X",             dict(gpu_name="MI355X")),
    ("+ speculation (α=0.75,k=4)", dict(spec_alpha=0.75)),
    ("context 4k → 16k",          dict(ctx=16384)),
    ("prefix hit 0% → 60%",       dict(prefix_hit=0.6)),
]

tornado = []
for label, delta in VARIATIONS:
    cfg = {**BASE, **delta}
    r = predict(**cfg)
    if "error" in r:
        print(f"{label:<30} infeasible: {r['error']}")
        continue
    tornado.append({"label": label,
                    "cpm": r["cpm"], "delta_pct": (r["cpm"] / base["cpm"] - 1) * 100,
                    "tps": r["decode_tps"], "tpot": r["tpot_ms"]})

tornado.sort(key=lambda d: d["delta_pct"])
print(f"{'change':<30}{'$/1M':>9}{'vs base':>10}{'tok/s':>11}{'TPOT':>9}")
print("-" * 70)
for t in tornado:
    print(f"{t['label']:<30}{t['cpm']:>8.3f}${t['delta_pct']:>+9.0f}%{t['tps']:>11,.0f}{t['tpot']:>8.1f}ms")

print("\nThe biggest cost lever is almost never the GPU brand.")

In [ ]:
JS = r'''
const M = {top: 16, right: 130, bottom: 40, left: 210};
const iw = W - M.left - M.right, ih = data.length * 26;
const svg = root.append("svg").attr("width",W).attr("height",ih+M.top+M.bottom)
    .append("g").attr("transform",`translate(${M.left},${M.top})`);
const ext = d3.max(data, d => Math.abs(d.delta_pct));
const x = d3.scaleLinear().domain([-ext*1.15, ext*1.15]).range([0, iw]);
const y = d3.scaleBand().domain(data.map(d=>d.label)).range([0, ih]).padding(0.2);
svg.append("g").attr("transform",`translate(0,${ih})`).call(d3.axisBottom(x).ticks(7)
   .tickFormat(d => d3.format("+d")(d) + "%"));
svg.append("text").attr("x",iw/2).attr("y",ih+34).attr("text-anchor","middle")
   .style("font-size","11.5px").text("change in $ per 1M output tokens vs baseline");
svg.append("g").call(d3.axisLeft(y).tickSize(0)).select(".domain").remove();
svg.selectAll("b").data(data).join("rect")
   .attr("x", d => d.delta_pct < 0 ? x(d.delta_pct) : x(0))
   .attr("y", d => y(d.label)).attr("height", y.bandwidth()).attr("rx",3)
   .attr("width", d => Math.abs(x(d.delta_pct) - x(0)))
   .attr("fill", d => d.delta_pct < 0 ? "#2e7d32" : "#c62828")
   .append("title").text(d => `${d.label}: $${d.cpm.toFixed(3)}/1M (${d3.format("+.0f")(d.delta_pct)}%)`);
svg.selectAll("t").data(data).join("text")
   .attr("x", d => (d.delta_pct < 0 ? x(d.delta_pct) - 6 : x(d.delta_pct) + 6))
   .attr("y", d => y(d.label) + y.bandwidth()/2 + 4)
   .attr("text-anchor", d => d.delta_pct < 0 ? "end" : "start")
   .style("font-size","10.5px").style("fill","#455a64")
   .text(d => `${d3.format("+.0f")(d.delta_pct)}%  ($${d.cpm.toFixed(3)})`);
svg.append("line").attr("x1",x(0)).attr("x2",x(0)).attr("y1",0).attr("y2",ih)
   .attr("stroke","#333").attr("stroke-width",1.5);
'''
show_d3(JS, tornado, height=len(tornado) * 26 + 60)

**Read the tornado.** Green bars are cheaper than baseline, red are more expensive. The
consistent pattern across almost every baseline you try:

1. **Batch size** (i.e. how close you run to the knee, [Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb)) dominates.
2. **Precision** is next, and it's nearly free to try.
3. **GPU choice** matters less than either — *once you've fixed the first two*. Teams routinely
   argue about #3 while leaving #1 misconfigured.

Note also that `context 4k → 16k` costs you nothing in `$/token` here but slashes
`max_concurrency` — a cost that shows up as *queueing* ([Reading the Logs](./Serving_Logs_Observability.ipynb), [Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb)), not as a per-token price. The
model reports it; the tornado can't show it. That's Part 7's point.

## Part 5 · The Pareto frontier

Every feasible configuration, plotted cost vs latency. A config is **dominated** if something else
is both cheaper *and* faster — those are greyed out. What remains is the real menu of choices.

In [ ]:
FRONT_MODEL, FRONT_CTX = "Llama-3.1-8B", 4096
pts = [d for d in grid if d["model"] == FRONT_MODEL and d["ctx"] == FRONT_CTX]

def pareto(points):
    '''Non-dominated set: minimize both cpm and tpot_ms.'''
    front = []
    for p in points:
        if not any((q["cpm"] <= p["cpm"] and q["tpot_ms"] <= p["tpot_ms"]) and
                   (q["cpm"] < p["cpm"] or q["tpot_ms"] < p["tpot_ms"]) for q in points):
            front.append(p)
    return sorted(front, key=lambda d: d["cpm"])

front = pareto(pts)
for p in pts:
    p["on_front"] = any(f is p for f in front)

print(f"{len(pts)} feasible configs for {FRONT_MODEL} at {FRONT_CTX} context; "
      f"{len(front)} are Pareto-optimal.\n")
print(f"{'GPU':<12}{'prec':<7}{'batch':>6}{'TP':>4}{'tok/s':>10}{'TPOT':>9}{'$/1M':>9}")
print("-" * 58)
for p in front[:14]:
    print(f"{p['gpu']:<12}{p['precision']:<7}{p['req_batch']:>6}{p['tp']:>4}"
          f"{p['decode_tps']:>10,.0f}{p['tpot_ms']:>8.1f}ms{p['cpm']:>8.3f}$")

print(f"\nEverything NOT on this list is a strictly worse deal than something that is.")
print("If your current production config isn't on the frontier, you're paying for nothing.")

In [ ]:
JS = r'''
const M = {top: 18, right: 20, bottom: 46, left: 66};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom;
const svg = root.append("svg").attr("width",W).attr("height",H).append("g")
    .attr("transform",`translate(${M.left},${M.top})`);
const x = d3.scaleLog().domain(d3.extent(data.pts, d=>d.tpot_ms)).range([0,iw]).nice();
const y = d3.scaleLog().domain(d3.extent(data.pts, d=>d.cpm)).range([ih,0]).nice();
svg.append("g").attr("transform",`translate(0,${ih})`).call(d3.axisBottom(x).ticks(6,"~s"));
svg.append("g").call(d3.axisLeft(y).ticks(6,"$~f"));
svg.append("text").attr("x",iw/2).attr("y",ih+38).attr("text-anchor","middle")
   .style("font-size","12px").text("TPOT — per-token latency (ms, log) →  slower");
svg.append("text").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-48)
   .attr("text-anchor","middle").style("font-size","12px")
   .text("$ per 1M output tokens (log) →  pricier");

svg.selectAll("d").data(data.pts).join("circle")
   .attr("cx",d=>x(d.tpot_ms)).attr("cy",d=>y(d.cpm))
   .attr("r",d=>d.on_front?5:2.6)
   .attr("fill",d=> !d.on_front ? "#cfd8dc" : (d.vendor==="NVIDIA" ? "#76b900" : "#ed1c24"))
   .attr("opacity",d=>d.on_front?1:0.5)
   .attr("stroke",d=>d.on_front?"#263238":"none").attr("stroke-width",0.8)
   .append("title").text(d=>`${d.gpu} · ${d.precision} · batch ${d.req_batch} · TP=${d.tp}\n` +
      `${d3.format(",.0f")(d.decode_tps)} tok/s · TPOT ${d.tpot_ms.toFixed(1)}ms · $${d.cpm.toFixed(3)}/1M` +
      (d.on_front ? "\n★ Pareto-optimal" : "\n(dominated)"));

const line = d3.line().x(d=>x(d.tpot_ms)).y(d=>y(d.cpm));
svg.append("path").datum(data.front.slice().sort((a,b)=>a.tpot_ms-b.tpot_ms))
   .attr("fill","none").attr("stroke","#263238").attr("stroke-width",1.4)
   .attr("stroke-dasharray","4 3").attr("d",line);
svg.append("text").attr("x",8).attr("y",14).style("font-size","11.5px").style("fill","#455a64")
   .text("grey = dominated · green = NVIDIA on the frontier · red = AMD on the frontier");
'''
show_d3(JS, {"pts": pts, "front": front}, height=380)

**The shape of this cloud is the honest answer to "which GPU should we buy?":** there isn't one
answer, there's a *frontier*, and where you sit on it is a business decision about how much latency
your product can tolerate. Both vendors appear on it — at different points.

## Part 6 · Six what-ifs, answered

Each of these is a question someone will actually ask you. Now you can answer with a number.

In [ ]:
def q(title, cfg_a, cfg_b, label_a="current", label_b="proposed"):
    a, b = predict(**cfg_a), predict(**cfg_b)
    print(f"\n{'='*78}\n{title}\n{'='*78}")
    for lbl, r in ((label_a, a), (label_b, b)):
        if "error" in r:
            print(f"  {lbl:<12} ❌ {r['error']}"); continue
        print(f"  {lbl:<12} {r['gpu']:<11} {r['precision']:<5} TP={r['tp']} · "
              f"{r['decode_tps']:>7,.0f} tok/s · TPOT {r['tpot_ms']:>5.1f}ms · "
              f"${r['cpm']:.3f}/1M · {r['bound']}-bound")
    if "error" not in a and "error" not in b:
        print(f"  → throughput {b['decode_tps']/a['decode_tps']:.2f}x · "
              f"cost {b['cpm']/a['cpm']:.2f}x · latency {b['tpot_ms']/a['tpot_ms']:.2f}x")
    return a, b

BASE8 = dict(model_name="Llama-3.1-8B", precision="fp16", batch=32, ctx=4096)

q("WHAT IF we moved our 8B fleet from A100 to MI300X?",
  {**BASE8, "gpu_name": "A100 80GB"}, {**BASE8, "gpu_name": "MI300X"}, "A100 80GB", "MI300X")

q("WHAT IF we quantized to FP8 instead of buying newer GPUs?",
  {**BASE8, "gpu_name": "H100 SXM"}, {**BASE8, "gpu_name": "H100 SXM", "precision": "fp8"},
  "H100 fp16", "H100 fp8")

q("WHAT IF we tried FP8 on our existing A100s? (the trap)",
  {**BASE8, "gpu_name": "A100 80GB"}, {**BASE8, "gpu_name": "A100 80GB", "precision": "fp8"},
  "A100 fp16", "A100 fp8")

q("WHAT IF we serve 70B instead of 8B — which vendor's topology wins?",
  dict(gpu_name="H100 SXM", model_name="Llama-3.1-70B", precision="fp16", batch=32, ctx=4096),
  dict(gpu_name="MI300X",   model_name="Llama-3.1-70B", precision="fp16", batch=32, ctx=4096),
  "H100 (TP)", "MI300X (TP=1)")

q("WHAT IF our context grows from 4k to 16k?",
  {**BASE8, "gpu_name": "H100 SXM"}, {**BASE8, "gpu_name": "H100 SXM", "ctx": 16384},
  "4k context", "16k context")

q("WHAT IF we turn on speculative decoding at low load?",
  dict(gpu_name="H100 SXM", model_name="Llama-3.1-8B", precision="fp16", batch=4, ctx=4096),
  dict(gpu_name="H100 SXM", model_name="Llama-3.1-8B", precision="fp16", batch=4, ctx=4096,
       spec_alpha=0.75, spec_k=4),
  "batch 4 plain", "batch 4 + spec")

print("\n\nNote the 16k-context answer: $/token barely moves, but max_concurrency collapses.")
print("The cost is real and shows up as QUEUEING (logs, benchmarking), which a per-token price cannot express.")

## Part 7 · Where this model lies to you

A planning model that doesn't publish its error bars is dangerous. Here is exactly what this one
ignores, and which notebook handles the reality instead:

| Ignored | Real effect | Where it's handled properly |
|---|---|---|
| **Queueing** | past the knee, latency explodes non-linearly; this model is linear | [Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb) (open-loop simulation) |
| **Prefill/decode interference** | prefill chunks steal decode slots; TTFT and TPOT interact | [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb), [Reading the Logs](./Serving_Logs_Observability.ipynb) |
| **Preemption & KV fragmentation** | throughput collapses when the pool saturates | [vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb), [Reading the Logs](./Serving_Logs_Observability.ipynb) |
| **Kernel quality per shape** | small-batch GEMMs run far below peak; varies by vendor and dtype | [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) Part 7, [Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb) |
| **Accuracy loss** | quantization is priced as free here; it isn't always | [Quantized Serving Showdown](./Quantized_Serving_Showdown.ipynb) |
| **Real prices** | spot/committed/regional pricing swings more than the numbers here | your invoice |
| **Multi-tenancy** | LoRA adapters, noisy neighbours, per-tenant SLOs | [Serving LoRA Adapters at Scale](./MultiLoRA_Serving_At_Scale.ipynb) |
| **Sequence-length variance** | assumes uniform lengths; real traffic is long-tailed | [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb), [Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb) |

**How to use it responsibly:**

1. Use the console to **shortlist** two or three configurations — that's what it's good at.
2. **Benchmark the shortlist for real** with [Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb)'s open-loop harness on the actual hardware.
3. Use [Reading the Logs](./Serving_Logs_Observability.ipynb)'s metrics to confirm the bottleneck the model predicted is the one you observe.
4. Re-calibrate Part 2's efficiency constants (`BW_EFF`, `FLOP_EFF`) from [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb)'s Part 7
   microbenchmark **on your own hardware**, then re-run.

That loop — model → shortlist → measure → recalibrate — is the actual job.

## Recap — the whole track, consolidated

| Question | The answer this track gives you |
|---|---|
| Why is decode slow? | Memory-bound; AI ≈ batch, left of every ridge point ([Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb), [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb)) |
| What's my ceiling? | KV pool ÷ context = concurrency; read it from the startup log ([Reading the Logs](./Serving_Logs_Observability.ipynb)) |
| Which optimization first? | Prompt layout → context size → quantization → batch tuning → speculation ([Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb)) |
| Is it working? | KV usage and queue depth lead; TTFT lags ([Reading the Logs](./Serving_Logs_Observability.ipynb)) |
| Which GPU? | Wherever your latency target meets the Pareto frontier — both vendors are on it ([The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb)–33) |
| Will my config port? | Not automatically: different ridge points, different kernel maturity ([The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb), [Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb)) |
| What will it cost? | Goodput and $/1M tokens, calibrated and error-barred ([Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb), [The What-If Console](./Serving_WhatIf_Console.ipynb)) |

### Further reading
- Everything this model consolidates: notebooks [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb)–[Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb)
- [Roofline model](https://dl.acm.org/doi/10.1145/1498765.1498785) · [DistServe (goodput as the objective)](https://arxiv.org/abs/2401.09670)
- [vLLM benchmarks](https://github.com/vllm-project/vllm/tree/main/benchmarks) — the measurement half of this loop

🏁 **End of the serving track.** [Back to the learning path](README.md).